In [ ]:
# Esto es para construir los modelos
from tensorflow.keras.layers import Dense, Dropout, Input, Flatten, Conv2D ,MaxPooling2D, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import plot_model


# Esto es para entrenamientos
from keras.callbacks import EarlyStopping
from livelossplot import PlotLossesKeras

# Esto es para evaluación

import numpy as np
import pandas as pd
# Graficos mmatrices
import plotly.express as px


In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

SEED = 161105

def set_seeds(seed_value=SEED):
    # 1. Set the PYTHONHASHSEED environment variable at a fixed value
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    # 2. Set the python built-in pseudo-random generator at a fixed value
    random.seed(seed_value)
    # 3. Set the numpy pseudo-random generator at a fixed value
    np.random.seed(seed_value)
    # 4. Set the tensorflow pseudo-random generator at a fixed value
    tf.random.set_seed(seed_value)

set_seeds()

In [1]:
from src import data_loader as dl
from src import EDA
from src import utils
from src import training as trm

# Carga de datos

In [2]:
X, Y = dl.load_data(fetch = False)

In [3]:
for tag in X.keys():
    print(X[tag].shape, Y[tag].shape)

(7007, 28, 28, 3) (7007, 1)
(2005, 28, 28, 3) (2005, 1)
(1003, 28, 28, 3) (1003, 1)


# Analisis Exploratorio

In [ ]:
fig1 = EDA.pixels_report(X = X)
fig1.show()

In [ ]:
fig2 = EDA.labels_report(Y = Y)
fig2.show()

## Escalamiento

In [5]:
X = utils.scaler.apply_scaler(X)   # note the () — you were missing the call}

In [ ]:
fig1 = EDA.pixels_report(X = X)
fig1.show()

In [6]:
df_imbalance, summary = EDA.imbalance_report(Y, key="train")
display(df_imbalance)
print(summary)

,count,percentage,ratio_vs_max,is_minority,is_rare
0,228,3.25,0.049,True,True
1,359,5.12,0.076,True,True
2,769,10.97,0.164,True,False
3,80,1.14,0.017,True,True
4,779,11.12,0.166,True,False
5,4693,66.98,1.000,False,False
6,99,1.41,0.021,True,True


{'total_samples': 7007, 'n_classes': 7, 'max_count': 4693, 'min_count': 80, 'imbalance_ratio_max_min': np.float64(58.66)}


In [7]:
X_aug, Y_aug = trm.augment_minority_classes(X, Y, key="train", n_minority=3)
# Verify the effect
df_after, summary_after = EDA.imbalance_report(Y_aug, key="train")
display(df_after)
print(summary_after)

Augmenting classes: [3 6 0] (counts: [ 80  99 228])


,count,percentage,ratio_vs_max,is_minority,is_rare
0,456,6.15,0.097,True,True
1,359,4.84,0.076,True,True
2,769,10.37,0.164,True,False
3,160,2.16,0.034,True,True
4,779,10.51,0.166,True,False
5,4693,63.30,1.000,False,False
6,198,2.67,0.042,True,True


{'total_samples': 7414, 'n_classes': 7, 'max_count': 4693, 'min_count': 160, 'imbalance_ratio_max_min': np.float64(29.33)}


In [8]:
X_aug.keys()

dict_keys(['train', 'test', 'validation'])

In [ ]:
fig2 = EDA.labels_report(Y = Y_aug)
fig2.show()

In [9]:
class_weights = trm.compute_class_weights(Y_aug, key="train")
print(class_weights)

{0: np.float64(2.322681704260652), 1: np.float64(2.950258654994031), 2: np.float64(1.3772989039569015), 3: np.float64(6.619642857142857), 4: np.float64(1.359618558591601), 5: np.float64(0.22568567166905118), 6: np.float64(5.349206349206349)}


In [10]:
import json
with open('./src/utils/convolutional.json', 'r') as file:
        # Deserialize the file content into a Python object (usually a dictionary or list)
        attributes = json.load(file)
model = trm.skin_classifier(X_aug,Y_aug ,attributes = attributes )


You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


In [ ]:
model.fit(X_aug,Y_aug,'train','test', epochs = 100, use_live_loss = True)

Epoch 1/100
244/464 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - balanced_accuracy: 0.0877 - loss: 1.9193

In [ ]:
model.metrics_report(X,Y)

In [ ]:
def export_model(model, path, fmt="keras"):
    """
    Exports a trained Keras model to disk.

    Parameters
    ----------
    model : skin_classifier
        A fitted instance of the classifier.
    path  : str
        Destination path including filename, e.g. "models/my_model".
        Extension is added automatically based on fmt.
    fmt   : str
        "keras" (default, recommended) or "h5" (legacy).
    """
    import os

    valid_formats = ("keras", "h5")
    if fmt not in valid_formats:
        raise ValueError(f"fmt must be one of {valid_formats}, got '{fmt}'")

    if model.history is None:
        raise RuntimeError(
            "Model has not been trained yet. Call .fit() before exporting."
        )

    # Ensure destination directory exists
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    # Strip any extension the user may have included and re-attach the right one
    base = os.path.splitext(path)[0]
    full_path = f"{base}.{fmt}"

    model.model.save(full_path)
    print(f"Model saved to: {full_path}")

    return full_path

In [ ]:
#export_model(model= model, path = './models/dermClass')

In [ ]:
model